# 실습. AI 티 안 나는 글쓰기 ✍️

> **상황.** 요즘 이메일·보고서·공지 초안은 AI 로 쓰는 경우가 많습니다.
> 그런데 받는 사람이 "이거 AI 가 썼네" 하고 느끼는 순간, 내용까지 가볍게 읽힙니다.
>
> **오늘 할 일.** AI 초안에서 **AI 티가 나는 곳을 찾고**, 오늘 배운 **system prompt · 예시(Few-shot) · 환각 가드** 로
> **사람이 쓴 업무 글** 로 고칩니다.

목표는 탐지기를 속이는 게 아니라 **읽는 사람이 믿고 읽는 글** 입니다.
회사에 AI 사용을 밝히라는 규정이 있으면 그대로 따르세요.

---

**코드를 몰라도 됩니다.** 위에서부터 ▶ 실행하고 결과(출력)를 읽어 보세요. 5단계에서 내 글을 넣어 봅니다.

## 0단계. 준비

In [1]:
# ⚙️ [그냥 실행하세요] OpenAI 연결 + 질문 함수
import re
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-6-luna"   # 호출마다 reasoning_effort="none" 을 함께 넣는다 (교재 5.3)

def ask(user, system=None, temperature=0.7):
    """system(규칙) 과 user(요청) 를 보내고 답만 돌려준다."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": user})
    resp = client.chat.completions.create(
        model=MODEL, reasoning_effort="none", temperature=temperature, messages=messages)
    return resp.choices[0].message.content

print("준비 완료:", MODEL)

준비 완료: gpt-6-luna


## 1단계. AI 초안 — 어디서 "AI 티"가 날까?

AI 에게 *"사내 뉴스레터에 실을 글을 써줘. 주제: 우리 팀이 생성형 AI 도구를 업무에 도입해 보니 어땠는지."* 라고만 해서 받은 글입니다 (gpt-6-luna 답변 그대로).

**먼저 직접 읽고, 어색한 곳을 찾아보세요.** (2분)

In [2]:
초안 = """
## 생성형 AI, 우리 팀의 일하는 방식을 바꾸고 있을까요?

우리 팀은 최근 생성형 AI 도구를 업무에 활용해 보기 시작했습니다. 처음에는 “어떤 일을 맡길 수 있을까?”라는 호기심에서 출발했지만, 써 볼수록 AI는 일을 대신해 주는 만능 도구라기보다 **반복적인 작업을 덜어 주고 생각의 출발점을 마련해 주는 조력자**에 가깝다는 것을 알게 됐습니다.

회의록을 정리하거나 긴 자료의 핵심을 요약하고, 이메일과 문서의 초안을 만드는 일은 특히 편리했습니다. 빈 화면 앞에서 오래 고민하는 대신 초안을 빠르게 만든 뒤, 팀의 맥락과 목적에 맞게 다듬을 수 있었죠. 아이디어를 여러 방향으로 펼쳐 보거나, 복잡한 내용을 다른 표현으로 설명해 달라고 요청하는 것도 유용했습니다.

물론 결과를 그대로 믿고 사용할 수는 없습니다. AI가 그럴듯하지만 틀린 정보를 내놓을 때도 있고, 우리 회사의 상황이나 업무 맥락을 충분히 이해하지 못할 때도 있습니다. 그래서 사실 확인과 최종 판단은 사람이 맡고 있습니다. 개인정보나 회사의 민감한 정보는 입력하지 않는 등 기본적인 보안 원칙도 지켜야 합니다.

이번 경험을 통해 얻은 교훈은 간단합니다. **AI를 잘 쓰는 일은 좋은 질문을 던지고, 결과를 꼼꼼히 검토하는 일**이라는 것입니다. 무엇을 맡기고 무엇을 직접 판단할지 구분할수록 도구의 장점도 더 잘 살릴 수 있습니다.

생성형 AI는 우리 팀의 일하는 방식을 한 번에 바꾸기보다, 작은 업무부터 조금씩 개선하고 있습니다. 앞으로도 유용했던 활용법과 주의할 점을 함께 나누며, 우리에게 맞는 방식으로 똑똑하게 활용해 보겠습니다.
""".strip()

print(초안)

## 생성형 AI, 우리 팀의 일하는 방식을 바꾸고 있을까요?

우리 팀은 최근 생성형 AI 도구를 업무에 활용해 보기 시작했습니다. 처음에는 “어떤 일을 맡길 수 있을까?”라는 호기심에서 출발했지만, 써 볼수록 AI는 일을 대신해 주는 만능 도구라기보다 **반복적인 작업을 덜어 주고 생각의 출발점을 마련해 주는 조력자**에 가깝다는 것을 알게 됐습니다.

회의록을 정리하거나 긴 자료의 핵심을 요약하고, 이메일과 문서의 초안을 만드는 일은 특히 편리했습니다. 빈 화면 앞에서 오래 고민하는 대신 초안을 빠르게 만든 뒤, 팀의 맥락과 목적에 맞게 다듬을 수 있었죠. 아이디어를 여러 방향으로 펼쳐 보거나, 복잡한 내용을 다른 표현으로 설명해 달라고 요청하는 것도 유용했습니다.

물론 결과를 그대로 믿고 사용할 수는 없습니다. AI가 그럴듯하지만 틀린 정보를 내놓을 때도 있고, 우리 회사의 상황이나 업무 맥락을 충분히 이해하지 못할 때도 있습니다. 그래서 사실 확인과 최종 판단은 사람이 맡고 있습니다. 개인정보나 회사의 민감한 정보는 입력하지 않는 등 기본적인 보안 원칙도 지켜야 합니다.

이번 경험을 통해 얻은 교훈은 간단합니다. **AI를 잘 쓰는 일은 좋은 질문을 던지고, 결과를 꼼꼼히 검토하는 일**이라는 것입니다. 무엇을 맡기고 무엇을 직접 판단할지 구분할수록 도구의 장점도 더 잘 살릴 수 있습니다.

생성형 AI는 우리 팀의 일하는 방식을 한 번에 바꾸기보다, 작은 업무부터 조금씩 개선하고 있습니다. 앞으로도 유용했던 활용법과 주의할 점을 함께 나누며, 우리에게 맞는 방식으로 똑똑하게 활용해 보겠습니다.


### AI 티 검사기 — 1일차 **정규식** 으로 만든 체크리스트

AI 글에 자주 나오는 버릇을 정규식으로 모았습니다. 점수가 높을수록 AI 티가 많습니다.

> ⚠️ 이건 **문체 체크리스트** 지 "AI 탐지기"가 아닙니다. 시중 AI 탐지기는 오탐이 많아서, OpenAI 도 2023년에 자체 탐지기를 정확도 문제로 내렸습니다.

In [3]:
# ⚙️ [그냥 실행하세요] AI 티 체크리스트 (이름 → 정규식)
AI_TI = {
    "굵은 글씨":     r"\*\*[^*\n]+\*\*",
    "제목·글머리표": r"(?m)^[ \t]*(#{1,6} |[-*•] |\d+\. )",
    "이모지":        r"[\U0001F300-\U0001FAFF☀-➿]",
    "긴 줄표 —":     r"—",
    "상투어":        r"다양한|효과적|중요한 역할|핵심적|혁신적|원활한|극대화|시너지|한층|인사이트|의미 있는|기반을 마련|기여|조력자|여정|살펴보|알아보",
    "대비 수사":     r"라기보다|가 아니라|이 아니라|뿐만 아니라|물론",
    "번역투":        r"에 있어|[을를] 통해|하는 것이 중요|가지고 있|것입니다",
    "교훈·다짐":     r"결론적으로|요약하자면|종합하면|간단합니다|교훈은|앞으로도|도움이 되|궁금한 점|언제든",
}

def 점수(글):
    return sum(len(re.findall(패턴, 글)) for 패턴 in AI_TI.values())

def 검사(글):
    """걸린 표현을 보여주고 점수를 출력한다."""
    for 이름, 패턴 in AI_TI.items():
        찾음 = [m.group().strip() for m in re.finditer(패턴, 글)]
        if 찾음:
            print(f"  {이름:<8} {len(찾음)}개  {찾음[:4]}")
    print(f"→ AI 티 점수: {점수(글)}   (글자 수 {len(글)})")

검사(초안)

  굵은 글씨    2개  ['**반복적인 작업을 덜어 주고 생각의 출발점을 마련해 주는 조력자**', '**AI를 잘 쓰는 일은 좋은 질문을 던지고, 결과를 꼼꼼히 검토하는 일**']
  제목·글머리표  1개  ['##']
  상투어      1개  ['조력자']
  대비 수사    2개  ['라기보다', '물론']
  번역투      2개  ['을 통해', '것입니다']
  교훈·다짐    3개  ['교훈은', '간단합니다', '앞으로도']
→ AI 티 점수: 11   (글자 수 799)


👉 그런데 체크리스트에 안 걸리는 더 큰 문제가 있습니다.
**도구 이름, 숫자, 실제 있었던 일이 하나도 없습니다.** 어느 회사 어느 팀에 붙여도 말이 되는 글입니다. → 4단계에서 다룹니다.

## 2단계. "자연스럽게 바꿔줘" 만으로는?

가장 흔한 방법부터 해봅니다.

In [4]:
A = ask("아래 글을 AI가 쓴 티 안 나게 자연스럽게 바꿔줘.\n\n" + 초안)
print(A)
print()
검사(A)

## 생성형 AI, 우리 팀의 일하는 방식을 바꾸고 있을까요?

우리 팀은 얼마 전부터 생성형 AI를 업무에 조금씩 활용하고 있습니다. 처음에는 “이걸로 뭘 할 수 있을까?” 하는 궁금증이 컸는데요. 직접 써 보니 AI가 일을 전부 대신해 주기보다는, 반복적인 작업을 줄이고 생각을 시작하는 데 도움을 주는 도구에 가까웠습니다.

회의록을 정리하거나 긴 자료를 요약하고, 이메일이나 문서의 초안을 만드는 일은 확실히 수월해졌습니다. 빈 화면을 붙들고 고민하는 대신 초안을 먼저 받아 본 뒤, 우리 팀의 상황과 목적에 맞게 고치면 되니까요. 아이디어를 여러 방향으로 넓혀 보거나 복잡한 내용을 쉽게 풀어 달라고 할 때도 꽤 유용했습니다.

그렇다고 AI가 내놓은 답을 그대로 믿을 수는 없습니다. 그럴듯해 보여도 틀린 정보가 섞여 있을 수 있고, 회사의 상황이나 업무 맥락을 제대로 반영하지 못할 때도 있습니다. 그래서 사실을 확인하고 최종 판단을 내리는 일은 사람이 해야 합니다. 개인정보나 회사의 민감한 내용은 입력하지 않는 등 보안에도 신경 쓰고 있습니다.

써 보면서 느낀 건, AI를 잘 활용하려면 질문을 구체적으로 하고 결과를 꼼꼼히 살펴야 한다는 점입니다. AI에 맡길 일과 직접 판단할 일을 구분할수록 도움을 더 잘 받을 수 있습니다.

생성형 AI가 우리 팀의 일하는 방식을 단번에 바꾼 건 아닙니다. 다만 작은 일부터 하나씩 편리해지고 있습니다. 앞으로도 실제로 도움이 된 방법과 주의할 점을 함께 나누면서, 우리 팀에 맞는 활용법을 찾아가려 합니다.

  제목·글머리표  1개  ['##']
  교훈·다짐    1개  ['앞으로도']
→ AI 티 점수: 2   (글자 수 765)


👉 말투만 조금 바뀌고 **길이·구조·결론은 그대로** 입니다. "자연스럽게"가 뭔지 AI 는 모릅니다.
→ 4교시에 배운 대로 **구체적인 규칙** 을 줘야 합니다.

## 3단계. 체크리스트를 **system prompt** 로

"좋은 System Prompt 작성법" 그대로입니다: 역할 + 금지 목록 + 형식 + **모르면 지어내지 말 것**.

In [5]:
편집_규칙 = """
너는 AI가 쓴 초안을 '사람이 직접 쓴 업무 글'로 고치는 편집자다.
규칙:
1. 마크다운 금지. 굵은 글씨, 제목(#), 글머리표 없이 평범한 문단으로 쓴다.
2. 이모지, 긴 줄표(—) 금지.
3. 금지 표현: 다양한, 효과적, 핵심적, 혁신적, 원활한, 인사이트, 의미 있는, 기반을 마련, 기여, 조력자, 여정, ~을 통해, ~에 있어, ~하는 것이 중요합니다, ~것입니다, 물론, ~라기보다, A가 아니라 B, 결론적으로, 교훈은 간단합니다, 앞으로도 ~하겠습니다.
4. 교훈이나 다짐으로 끝내지 않는다. 할 말을 다 하면 그냥 끝낸다.
5. 일반론은 지우고, 이 팀만 할 수 있는 이야기만 남긴다. 전체 길이는 절반 정도로 줄인다.
6. 문장 길이를 섞는다. 짧은 문장도 넣는다.
7. 구체적인 사실(도구 이름, 숫자, 사례, 날짜)이 들어가야 더 설득력 있는 자리에는 지어내지 말고 [ ] 로 빈칸을 남긴다. 예: [줄어든 시간], [실제 사례]. 빈칸은 2~4개.
고친 글만 출력한다.
""".strip()

B = ask(초안, system=편집_규칙)
print(B)
print()
검사(B)

우리 팀은 회의록 정리, 긴 자료 요약, 이메일과 문서 초안 작성에 생성형 AI를 써 보고 있습니다. 빈 화면에서 시작하는 시간은 줄었지만, 결과를 그대로 쓰지는 않습니다. 회사 맥락에 맞게 고치고 사실을 확인하는 일은 여전히 팀의 몫입니다.

최근 [실제 사례]를 정리할 때는 초안을 먼저 만든 뒤 담당자가 내용을 검토했습니다. 다만 AI가 틀린 정보를 그럴듯하게 제시하거나 업무 맥락을 놓치는 경우도 있었습니다. 개인정보와 회사의 민감한 정보는 입력하지 않습니다.

우리 팀에서 AI가 맡는 일은 반복 작업과 초안 작성입니다. 최종 판단은 사람이 합니다. [사용 도구]를 활용한 뒤 [줄어든 시간]만큼 검토와 수정에 시간을 쓰고 있습니다.

→ AI 티 점수: 0   (글자 수 359)


👉 점수는 0 에 가까워집니다. 그런데 읽어 보면 **밋밋하고, [ ] 빈칸** 이 생겼습니다.

규칙 7번 덕분에 AI 가 **없는 사실을 지어내지 않고 빈칸으로 남긴 것** 입니다 (4교시 환각 가드).
**점수 0 ≠ 좋은 글.** 이 빈칸이 바로 **사람이 채워야 할 자리** 입니다.

## 4단계. **내 말투 예시 + 내 사실 메모** — Few-shot + 사실 주입

- **내 말투 예시** = Few-shot. 내가 실제로 쓴 짧은 글 2~3개를 보여주면 그 말투를 따라 합니다.
- **내 사실 메모** = 빈칸에 들어갈 진짜 내용. 메모에 없는 건 지어내지 말라고 못 박습니다.

In [6]:
내_말투 = """
내가 평소에 쓰는 글 예시 (이 말투를 따라라):
---
다들 고생 많으셨습니다. 배포는 오늘 밤 10시에 합니다. 문제 생기면 바로 저한테 전화 주세요.
---
지난주 장애 원인 찾았습니다. 캐시 만료 시간이 1분으로 잘못 들어가 있었어요. 오늘 고쳤고, 같은 설정이 다른 서버에도 있는지 내일까지 보겠습니다.
---
"""

내_사실 = """
[내 사실 메모 — 이 사실만 사용하고 더 지어내지 마라]
- 3월부터 팀원 6명이 같이 써 봄
- 주간보고 초안 쓰는 시간: 1인당 1시간쯤 → 20분 정도
- 회의록 요약은 거의 매번 씀. 숫자는 꼭 원본과 대조함
- 4월에 AI 초안에 고객사 이름이 틀리게 들어감. 메일 보내기 직전에 잡음
- 그 뒤로 고객 정보는 넣지 않기로 팀 규칙 정함
"""

D = ask(초안, system=편집_규칙 + "\n" + 내_말투 + 내_사실)
print(D)
print()
검사(D)

3월부터 팀원 6명이 생성형 AI를 같이 써 봤습니다. 주간보고 초안을 쓰는 시간은 1인당 1시간쯤에서 20분 정도로 줄었습니다. 회의록 요약도 거의 매번 씁니다. 숫자는 원본과 꼭 대조합니다.

4월에는 AI 초안에 고객사 이름이 틀리게 들어간 적이 있습니다. 메일을 보내기 직전에 발견했습니다. 그 뒤로 고객 정보는 입력하지 않기로 팀 규칙을 정했습니다.

초안은 빨리 만들 수 있습니다. 다만 그대로 보내지는 않습니다. [사용한 도구]에서 나온 내용을 확인하고, 팀 상황에 맞게 고칩니다.

→ AI 티 점수: 0   (글자 수 276)


In [9]:
변수_이름들 = [
    "MODEL", "초안", "AI_TI", "A",
    "편집_규칙", "B", "내_말투", "내_사실", "D",
    "비교", "내_초안", "내_말투2", "내_사실2", "결과",
]

for 이름 in 변수_이름들:
    print(f"\n--- {이름} ---")
    if 이름 in globals():
        print(globals()[이름])
    else:
        print("아직 선언 안 됨 — 해당 코드 셀을 먼저 실행하세요.")


--- MODEL ---
gpt-6-luna

--- 초안 ---
## 생성형 AI, 우리 팀의 일하는 방식을 바꾸고 있을까요?

우리 팀은 최근 생성형 AI 도구를 업무에 활용해 보기 시작했습니다. 처음에는 “어떤 일을 맡길 수 있을까?”라는 호기심에서 출발했지만, 써 볼수록 AI는 일을 대신해 주는 만능 도구라기보다 **반복적인 작업을 덜어 주고 생각의 출발점을 마련해 주는 조력자**에 가깝다는 것을 알게 됐습니다.

회의록을 정리하거나 긴 자료의 핵심을 요약하고, 이메일과 문서의 초안을 만드는 일은 특히 편리했습니다. 빈 화면 앞에서 오래 고민하는 대신 초안을 빠르게 만든 뒤, 팀의 맥락과 목적에 맞게 다듬을 수 있었죠. 아이디어를 여러 방향으로 펼쳐 보거나, 복잡한 내용을 다른 표현으로 설명해 달라고 요청하는 것도 유용했습니다.

물론 결과를 그대로 믿고 사용할 수는 없습니다. AI가 그럴듯하지만 틀린 정보를 내놓을 때도 있고, 우리 회사의 상황이나 업무 맥락을 충분히 이해하지 못할 때도 있습니다. 그래서 사실 확인과 최종 판단은 사람이 맡고 있습니다. 개인정보나 회사의 민감한 정보는 입력하지 않는 등 기본적인 보안 원칙도 지켜야 합니다.

이번 경험을 통해 얻은 교훈은 간단합니다. **AI를 잘 쓰는 일은 좋은 질문을 던지고, 결과를 꼼꼼히 검토하는 일**이라는 것입니다. 무엇을 맡기고 무엇을 직접 판단할지 구분할수록 도구의 장점도 더 잘 살릴 수 있습니다.

생성형 AI는 우리 팀의 일하는 방식을 한 번에 바꾸기보다, 작은 업무부터 조금씩 개선하고 있습니다. 앞으로도 유용했던 활용법과 주의할 점을 함께 나누며, 우리에게 맞는 방식으로 똑똑하게 활용해 보겠습니다.

--- AI_TI ---
{'굵은 글씨': '\\*\\*[^*\\n]+\\*\\*', '제목·글머리표': '(?m)^[ \\t]*(#{1,6} |[-*•] |\\d+\\. )', '이모지': '[\\U0001F300-\\U0001FAFF☀-➿]', '긴 줄표 —': '—', '상투어': 

👉 이제 **"우리 팀 이야기"** 가 됐습니다. 이 글도 AI 가 썼습니다. 사실을 줬을 뿐입니다.

**AI 티의 정체는 문장 버릇보다 "내용이 없다" 는 것** 입니다.

### 한눈에 비교

In [8]:
비교 = pd.DataFrame([
    {"단계": 이름, "AI 티 점수": 점수(글), "글자 수": len(글), "[ ] 빈칸": len(re.findall(r"\[[^\]]+\]", 글))}
    for 이름, 글 in [("1. AI 초안", 초안), ("2. 자연스럽게", A), ("3. 규칙", B), ("4. 규칙+말투+사실", D)]
])
비교

,단계,AI 티 점수,글자 수,[ ] 빈칸
0,1. AI 초안,11,799,0
1,2. 자연스럽게,2,765,0
2,3. 규칙,0,359,3
3,4. 규칙+말투+사실,0,276,1


## 5단계. ✍️ 내 글로 해보기 (25분)

1. 아래 셀의 **요청 문장을 내 업무로 바꿔** AI 초안을 받습니다. (이미 AI 로 쓴 글이 있으면 `내_초안 = """ ... """` 로 붙여넣기)
2. `검사` 로 AI 티 확인
3. **내 말투 예시 · 내 사실 메모** 를 내 것으로 바꿔 다듬기
4. 마지막은 사람이 — 아래 **최종 손질 3가지**

> ⚠️ 회사 기밀·고객 정보는 넣지 마세요. 사실 메모도 마찬가지입니다.
> 막히면 손 들기 — 문장만 바꾸면 되는 셀이라 코드를 몰라도 괜찮습니다.

In [23]:
# ✏️ 요청 문장을 내 업무로 바꿔 실행
#    예: 협력사에 납기 지연 사과 메일 / 신규 입사자 환영 메일 / 분기 회고 보고서 / 고객 공지
내_초안 = ask("강사님에게 말씀드릴말 좀 6하원칙에 따라 친근하게 써줘. 수업을 좀 빨리 끝내고 싶어.")
print(내_초안)
print()
검사(내_초안)

강사님, 혹시 오늘 수업을 **조금 일찍 마쳐도 괜찮을까요?**  
제가 **[이유]** 때문에 **[원하는 시간]쯤** 먼저 이동해야 할 것 같아서요. 수업에 지장이 없도록 **[미리 준비할 내용이나 보충 방법]**은 챙기겠습니다. 갑작스럽게 말씀드려 죄송하고, 가능하신지 편하게 알려주세요!

- **누가:** 제가  
- **언제:** 오늘  
- **어디서:** 수업에서  
- **무엇을:** 수업을 조금 일찍 마치고 싶습니다  
- **왜:** [이유] 때문에  
- **어떻게:** [원하는 시간]쯤 마치고, 필요한 내용은 보충하겠습니다  

`[이유]`와 `[원하는 시간]`만 상황에 맞게 바꿔서 보내면 돼요.

  굵은 글씨    10개  ['**조금 일찍 마쳐도 괜찮을까요?**', '**[이유]**', '**[원하는 시간]쯤**', '**[미리 준비할 내용이나 보충 방법]**']
  제목·글머리표  6개  ['-', '-', '-', '-']
→ AI 티 점수: 16   (글자 수 349)


In [26]:
# ✏️ 내가 실제로 쓴 메일·메신저 2~3개, 그리고 진짜 사실 메모로 바꿔 실행

내_말투2 = """
내가 평소에 쓰는 글 예시 (이 말투를 따라라):
---
아니 이게 무슨 일이냐는 거쥐~
내가 분명히 말했자나아 ㅋㅋㅋ
이거 완전 레전드인 부분이구우~
오히려 좋아~ 완전 럭키비키잖아아~
그건 좀 아닌 거 줴줴이야~
어쩔 수 없는 부분인 거쥐~
---
말투 규칙:
1. 친한 친구에게 장난치듯 능청스럽게 말한다.
2. 문장 끝에 "~쥐", "~자나아", "~라구우", "~거야아" 등을 자연스럽게 사용한다.
3. "줴줴이야~"를 2회 이상 무조건 사용한다.
4. ㅋㅋㅋ, ㅠㅠ, ~ 등을 적절하게 사용한다.
5. 모든 문장을 억지로 변형하지 않는다.
6. 원문의 의미와 사실관계는 절대 바꾸지 않는다.
"""

내_사실2 = """
[내 사실 메모 — 이 사실만 사용하고 더 지어내지 마라]
"""

결과 = ask(내_초안, system=편집_규칙 + "\n" + 내_말투2 + 내_사실2)
print(결과)
print()
검사(결과)

강사님, 오늘 [이유] 때문에 [원하는 시간]쯤 먼저 이동해야 할 것 같아요. 수업에 지장 없게 [미리 준비할 내용이나 보충 방법]은 챙길게요. 갑자기 말씀드려 죄송해요 ㅠㅠ 가능하신지 알려주시면 감사하겠쥐. 줴줴이야~ 제가 너무 급하게 말했쥐? 줴줴이야~

→ AI 티 점수: 0   (글자 수 143)


In [ ]:
# ✏️ (선택) 내가 발견한 AI 버릇을 체크리스트에 추가 — 여러 개는 | 로 잇기
AI_TI["내가 찾은 것"] = r"진심으로|기대가 큽니다|함께 만들어"
검사(결과)

### 최종 손질 3가지 — 사람만 할 수 있는 일

1. **사실 확인** — 숫자·이름·날짜를 원본과 대조합니다. (4단계 예시의 "고객사 이름 사고"처럼)
2. **소리 내어 읽기** — 내가 말로는 안 할 문장이면 고치거나 지웁니다.
3. **마지막 문단 지워 보기** — 지워도 뜻이 통하면 지웁니다. AI 는 꼭 다짐으로 끝냅니다.

## 📌 오늘의 정리

| | 핵심 메시지 |
|---|---|
| 🔍 **AI 티의 정체** | 굵은 글씨·"~라기보다"·다짐 마무리도 있지만, 진짜는 **구체적인 내용이 없다** 는 것 |
| 🧰 **고치는 법** | "자연스럽게" ✗ → **금지 목록**(system prompt) + **내 말투 예시**(Few-shot) + **내 사실 메모**(환각 가드) |
| ✋ **사람 몫** | 사실을 주고, 확인하고, 책임지는 건 사람. 모르는 건 AI 가 **빈칸으로 남기게** 하는 게 안전 |

> **다음 (3일차).** 오늘은 사실 메모를 손으로 넣었습니다. 내일 배우는 **RAG** 는 *내 문서에서 사실을 찾아 자동으로 넣는* 방법입니다.